# Redrob AI Ranker — Sandbox Demo

**India Runs Data & AI Challenge | Submission by AlfaizKureshi**

---

This notebook runs the full ranking pipeline end-to-end on a 50-candidate sample.

| What | Detail |
|---|---|
| Pipeline | Hard disqualify → 11-signal base score → behavioral multiplier → penalties |
| Runtime | ~5–15 seconds on this sample (vs ~25s on the full 100K dataset) |
| GPU | Not required — CPU only |
| Network | No external API calls during ranking |
| Output | Ranked CSV with candidate_id, rank, score, reasoning |

**To reproduce: Runtime → Run All (Ctrl+F9)**

In [ ]:
# ── Step 1: Clone the repo ──────────────────────────────────────────────────
# Fresh clone every run so the notebook is always reproducible.
!rm -rf redrob-ai-ranker
!git clone https://github.com/Alfaiz777/redrob-ai-ranker.git --quiet
%cd redrob-ai-ranker
print('[OK] Repository cloned — working directory set to redrob-ai-ranker')

In [ ]:
# ── Step 2: Install dependencies ────────────────────────────────────────────
# The ranking engine uses only Python stdlib (json, re, multiprocessing, csv).
# We only install pandas here for displaying results in a readable table.
# Heavy packages (sentence-transformers, faiss-cpu) are for the optional FAISS
# retrieval pipeline — not needed for the standard ranking step.
!pip install pandas -q
print('[OK] pandas installed for result display')

In [ ]:
# ── Step 3: Prepare sample input ────────────────────────────────────────────
# The ranking pipeline expects JSONL (one JSON object per line).
# The repo ships a sample_candidates.json (JSON array) for readability.
# This cell converts it to JSONL so run.py can process it.
import json

with open('data/sample_candidates.json', 'r', encoding='utf-8') as f:
    candidates = json.load(f)

with open('data/sample_candidates.jsonl', 'w', encoding='utf-8') as f:
    for c in candidates:
        f.write(json.dumps(c) + '\n')

print(f'[OK] {len(candidates)} candidates converted to JSONL format')
print(f'     Sample candidate ID: {candidates[0]["candidate_id"]}')

In [ ]:
# ── Step 4: Run the full ranking pipeline ───────────────────────────────────
# This is the exact same command documented in README.md.
# On the full 100K dataset this takes ~25 seconds on 8 cores.
# On this 50-candidate sample it completes in a few seconds.
!python run.py --input data/sample_candidates.jsonl --out sample_submission.csv

In [ ]:
# ── Step 5: Display ranked results ──────────────────────────────────────────
import pandas as pd

df = pd.read_csv('sample_submission.csv')

print(f'Candidates ranked : {len(df)}')
print(f'Score range       : {df["score"].min():.4f}  to  {df["score"].max():.4f}')
print(f'Scores non-increasing: {all(df["score"].iloc[i] >= df["score"].iloc[i+1] for i in range(len(df)-1))}')
print()

pd.set_option('display.max_colwidth', 110)
display(df[['rank', 'candidate_id', 'score', 'reasoning']])

In [ ]:
# ── Step 6: Download the CSV ─────────────────────────────────────────────────
# Downloads sample_submission.csv to your local machine.
# Skip this cell if running outside Colab.
try:
    from google.colab import files
    files.download('sample_submission.csv')
    print('[OK] sample_submission.csv downloaded')
except ImportError:
    print('[INFO] Not running in Colab — find sample_submission.csv in the working directory')